# Workshop Databricks — Lakeflow Designer
### Guia passo a passo (PT-BR)

Bem-vindo(a)! Neste workshop você monta um **visual data prep** no Lakeflow Designer para a **Aurora Café**: qualidade de dados, união de canais, joins, IA em tickets de suporte e escrita no Unity Catalog.


**Agenda**

| # | Módulo | O que você vai fazer |
|---|---|---|
| 1 | Upload de CSVs | Subir 3 arquivos e criar tabelas |
| 2 | Canvas e fontes | Source, preview e parâmetro |
| 3 | Qualidade | Unique, Prepare, Guardrails, Filter, Combine |
| 4 | Relacionar | Join, Select, SQL com parâmetro |
| 5 | IA e join | AI Function, Join em `pedido_id`, Select |
| 6 | Publicar | Output, Group |
| 7 | Genie Code *(opcional)* | Recriar o fluxo em linguagem natural |
| 8 | Dashboard *(opcional)* | Prompt para um AI/BI Dashboard da tabela ouro |

Documentação: [operadores nativos](https://docs.databricks.com/aws/en/designer/built-in-operators) · [o que é o Designer](https://docs.databricks.com/aws/en/designer/what-is-lakeflow-designer)


## Descrição das tabelas

### `clientes` (82 linhas, 2 duplicatas de propósito)

| Coluna | Descrição |
|---|---|
| `cliente_id` | ID do cliente (duplicado em 2 linhas) |
| `nome` | Nome; alguns com espaços extras |
| `email` | Alguns vazios |
| `cidade` / `estado` | Grafia inconsistente em algumas cidades |
| `canal_aquisicao` | `APP`, `LOJA`, `PARCEIRO`, `SITE` |
| `segmento` | `Bronze`, `Prata`, `Ouro` |
| `data_cadastro` | Data de cadastro |
| `ativo` | `true` / `false` |

### `pedidos` (201 linhas)

| Coluna | Descrição |
|---|---|
| `pedido_id` | ID do pedido (há duplicata) |
| `cliente_id` | FK para clientes |
| `produto` / `categoria` | Item e categoria do cardápio |
| `quantidade` / `valor_unitario` / `valor_total` | Há um valor total negativo |
| `canal` | `ONLINE` ou `LOJA` (há variação de caixa/espaço) |
| `status` | `confirmado`, `pendente`, `cancelado` (há `PENDNETE`) |
| `data_pedido` / `loja` | Data e loja Aurora |

### `tickets_suporte` (60 linhas)

| Coluna | Descrição |
|---|---|
| `ticket_id` | ID do ticket |
| `cliente_id` / `pedido_id` | FKs |
| `comentario` | Texto em PT-BR para `ai_classify` |
| `sentimento_esperado` | Gabarito (`positivo`, `misto`, `negativo`) |


---
## Configuração inicial — widgets

Preencha catálogo, schema e o seu prefixo único (ex.: `ana_`).


In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("nome_catalogo", "seu_catalogo", "Catálogo")
dbutils.widgets.text("nome_schema", "seu_schema", "Schema")
dbutils.widgets.text("seu_prefixo", "", "Seu Prefixo (ex: ana_)")


In [0]:
nome_catalogo = dbutils.widgets.get("nome_catalogo")
nome_schema = dbutils.widgets.get("nome_schema")
seu_prefixo = dbutils.widgets.get("seu_prefixo")

print(f"Catálogo : {nome_catalogo}")
print(f"Schema   : {nome_schema}")
print(f"Prefixo  : {seu_prefixo}")


Catálogo : serverless_stable_x3nqbh_catalog
Schema   : hackathon_schema
Prefixo  : caroljf_


---
## Módulo 1 — Subindo arquivos manualmente (CSV)


### Passo a passo

1. No menu lateral, clique em **+ New** → **Add or upload data** (ou **Catalog** → **Create** → **Create table**).
2. Arraste o CSV da pasta `dados/` ou selecione do computador.
3. Revise o schema detectado (tipos das colunas).
4. Escolha **Catálogo**, **Schema** e o nome da tabela **com o seu prefixo**.
5. Clique em **Create table**.
6. Repita para os 3 arquivos.

### Arquivos obrigatórios

| Arquivo | Nome sugerido da tabela |
|---|---|
| `clientes.csv` | `{seu_prefixo}clientes` |
| `pedidos.csv` | `{seu_prefixo}pedidos` |
| `tickets_suporte.csv` | `{seu_prefixo}tickets_suporte` |

> Exemplo: prefixo `ana_` → tabela `ana_clientes`.

### Validação rápida

Depois do upload, confira no **Catalog** se as 3 tabelas aparecem e abra **Sample data**. Rode as células SQL abaixo.


In [0]:
%sql
SELECT COUNT(*) AS qtd_clientes
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}clientes`;


qtd_clientes
82


In [0]:
%sql
SELECT cliente_id, COUNT(*) AS ocorrencias
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}clientes`
GROUP BY cliente_id
HAVING COUNT(*) > 1
ORDER BY ocorrencias DESC;


cliente_id,ocorrencias
1,2
5,2


In [0]:
%sql
SELECT canal, status, COUNT(*) AS qtd, ROUND(SUM(valor_total), 2) AS receita
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}pedidos`
GROUP BY canal, status
ORDER BY canal, status;


canal,status,qtd,receita
LOJA,cancelado,13,367.5
LOJA,confirmado,39,1124.5
LOJA,pendente,14,409.5
Loja,confirmado,1,37.5
ONLINE,PENDNETE,1,19.8
ONLINE,cancelado,26,775.7
ONLINE,confirmado,80,2380
ONLINE,pendente,26,775.7
online,cancelado,1,19.8


In [0]:
%sql
SELECT sentimento_esperado, COUNT(*) AS qtd
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}tickets_suporte`
GROUP BY sentimento_esperado
ORDER BY qtd DESC;


sentimento_esperado,qtd
positivo,20
misto,20
negativo,20


**Checkpoint do Módulo 1**

- [ ] 3 tabelas no seu schema
- [ ] `clientes` tem mais de 80 linhas (duplicatas visíveis)
- [ ] `pedidos` tem canais ONLINE e LOJA
- [ ] `tickets_suporte` tem 60 comentários


---
## Módulo 2 — Canvas, Source, preview e parâmetro

### 2.1 Criar o visual data prep

1. Clique em **New** na barra lateral → **Visual data prep**.
2. Renomeie o arquivo no topo para `{seu_prefixo}aurora_performance`.
3. No seletor de compute, escolha **Serverless**.

Interface: painel de operadores à esquerda, canvas no centro, configuração à direita, preview embaixo, prompt do **Genie Code** acima do canvas. Ver [interface](https://docs.databricks.com/aws/en/designer/interface).

### 2.2 Source — três tabelas

O operador **Source** importa uma tabela do Unity Catalog (ou arquivo). Depois de selecionar, o painel mostra nome, dono e última atualização.

Para cada tabela:

1. No painel **Operators**, clique em **Source** (ou arraste a tabela do painel **Catalog**).
2. Busque `{seu_prefixo}clientes` e selecione. Renomeie o operador para `clientes`.
3. Repita para `pedidos` e `tickets_suporte`.
4. Clique em cada Source e confira o preview.

### 2.3 Preview e perfil dos dados

Antes de transformar, use o canvas para conhecer as fontes — sem digitar tabela.

1. Clique no Source `pedidos` e abra o **output pane** embaixo.
2. Confira colunas e algumas linhas. Você deve ver `canal` com variação (`ONLINE`, `LOJA`, `online `, `Loja`) e pelo menos um `status` estranho (`PENDNETE`).
3. Selecione a coluna `canal` (ou `status`) e abra o **data profile** para ver a distribuição dos valores. Essa sujeira é o que Unique, Prepare e Guardrails vão tratar no módulo 3.
4. Repita no Source `clientes`: busque duplicatas de `cliente_id` e e-mails vazios.
5. No painel esquerdo, abra **Table of contents** para listar os três Sources. Conforme o canvas crescer, é por aqui que você acha um operador pelo nome.


### 2.4 Parâmetro de cidade

1. Painel esquerdo → **Parameters** → adicionar parâmetro.
2. Nome: `cidade_filtro`
3. Tipo texto, valor padrão: `São Paulo`

Você vai usar esse parâmetro no operador **SQL** (`:cidade_filtro`).


---
## Módulo 3 — Qualidade: Unique, Prepare, Guardrails, Filter, Combine

Neste módulo você trata a sujeira **intencional** dos CSVs.

### 3.1 Unique em clientes

**Unique** remove duplicatas. Por padrão, compara todas as colunas; aqui a chave é `cliente_id`.

1. A partir de `clientes`, adicione **Unique**. Nome: `clientes_unicos`.
2. Unique by: **Selected columns** → `cliente_id`.
3. Order by: `email` DESC (mantém a linha que tem e-mail preenchido).
4. Preview: a duplicata some.

### 3.2 Prepare em clientes

**Prepare** encadeia ações de limpeza. Adicione **Prepare** depois de `clientes_unicos`. Nome: `clientes_limpos`.

Ações (nessa ordem):

| Ação | Configuração |
|---|---|
| Trim | coluna `nome` |
| Text case | `cidade` → Title (ou fórmula para padronizar) |
| Fill nulls | `email` → `sem_email@auroracafe.com` |
| Replace value | `cidade` valor `SAO PAULO` → `São Paulo` |
| Replace value | `cidade` valor `curitiba` → `Curitiba` |
### 3.3 Unique + Prepare em pedidos

1. **Unique** em `pedidos` → `pedidos_unicos`, chave `pedido_id`.
2. **Prepare** → `pedidos_preparados`:
   - Trim em `canal`
   - Text case **upper** em `canal`
   - Formula: nome da coluna nova `receita` formula `quantidade * valor_unitario` (não confie só em `valor_total`, há valor negativo)

### 3.4 Guardrails em pedidos

**Guardrails** valida regras e separa **Passed** / **Failed** / **Errors**.

1. Adicione **Guardrails** depois de `pedidos_preparados`. Nome: `qualidade_pedidos`.
2. When a check fails: **Warn and continue with passed rows**.
3. Checks:
   - **Not null**: `pedido_id`
   - **Not null**:  `cliente_id`
   - **Not null**:  `produto`
   - **Accepted values**: `status` ∈ `confirmado`, `pendente`, `cancelado`
   - **Value range**: `valor_total` ≥ 0
4. Inspecione as 3 saídas. O status `PENDNETE` e o valor negativo devem ir para **Failed**.
5. Siga o workshop pela saída **Passed**.

### 3.5 Filter — duas saídas (True/ False)

**Filter** não descarta o resto: **True** = quem passa; **False** = quem não passa (e nulos).

1. A partir de **Passed**, adicione **Filter**. Nome: `separa_canal`.
2. Condição: `canal` **Is equal to** `ONLINE`.
3. Preview:
   - True = pedidos online
   - False = pedidos de loja

### 3.6 Prepare em cada ramo + Combine (Union)

1. No **True**, Prepare `marca_online`: Formula Nova coluna `origem` valor 'ONLINE'`.
2. No **False**, Prepare `marca_loja`: Formula  Nova coluna `origem` valor 'LOJA'.
3. Adicione **Combine**. Nome: `uniao_canais`.
4. Ligue as duas saídas.
5. Set operation: **Union**.
6. Merge strategy: **All**.
7. Advanced: **Union by name** + **Allow missing columns**.



---
## Módulo 4 — Join, Select e SQL com parâmetro

### 4.1 Join pedidos × clientes

**Join** (2 tabelas) tem, por padrão, **3 saídas**: Left unmatched, Matched, Right unmatched. Para um único resultado, escolha o tipo (Inner / Left / …).

1. Adicione **Join**. Nome: `join_pedidos_clientes`.
2. Entrada 1: `uniao_canais`. Entrada 2: `clientes_limpos`.
3. Condição: `cliente_id` = `cliente_id`.
4. Join type: **Inner join** (uma saída só).
5. Match case: off (padrão).
6. Preview: cada pedido ganha cidade, segmento, canal de aquisição.


### 4.2 Select — enxugar colunas

**Select** escolhe, renomeia e reordena. Também aceita regra dinâmica (tipo, nome, fórmula em metadados).

1. Adicione **Select**. Nome: `colunas_analise`.
2. Clique em **Start with no columns** e marque:
   `pedido_id`, `cliente_id`, `produto`, `categoria`, `status`, `data_pedido`, `loja`, `receita`, `origem`, `nome`, `cidade`, `segmento`.
3. Renomeie `origem` → `canal_venda`.

### 4.3 SQL — filtro por parâmetro

O operador **SQL** cobre o que os blocos visuais não fazem. Referencie o operador de cima **pelo nome** e parâmetros com `:nome`.

1. Adicione **SQL**. Nome: `filtra_cidade`.
2. Query:

```sql
SELECT *
FROM colunas_analise
WHERE cidade = :cidade_filtro
   OR :cidade_filtro = 'TODAS'
```

3. Troque o parâmetro entre `São Paulo` e `TODAS` e veja o preview mudar — mesmo padrão do Sidney (`WHERE city = :city_filter`).

Não filtre `confirmado` neste canvas. O `status` segue no pedido e o Join de tickets usa `filtra_cidade` direto.


---
## Módulo 5 — AI Function, Join em `pedido_id`, Select

Crie um **Group** `Pipeline de sentimento` e coloque nele: `tickets_com_texto`, `classifica_sentimento`, `extrai_sentimento`, `experiencia_pedido`, `colunas_finais`.

### 5.1 Filter de texto

Source `tickets_suporte`. Filter: `comentario` **Is not null**. Nome: `tickets_com_texto`. Siga pela saída **True**.

### 5.2 AI Function — classificar sentimento

1. A partir do **True**, adicione **AI Function**. Nome: `classifica_sentimento`.
2. Function: `ai_classify`.
3. Input: `comentario`.
4. Labels: `positivo`, `misto`, `negativo`.
5. Output column: `sentimento`.
6. Preview (pode demorar alguns segundos).

O campo `sentimento` **não** é texto. O `ai_classify` (v2.0) devolve um VARIANT:

```json
{
  "response": ["negativo"],
  "metadata": { "version": "2.0" },
  "error_message": null
}
```

O rótulo útil está em `response[0]`. Não agrupe por `sentimento` ainda.

### 5.3 Prepare — extrair o rótulo

1. Adicione **Prepare**. Nome: `extrai_sentimento`.
2. Ação **Formula**:
   - Expression: `sentimento:response[0]::STRING`
   - Create a new column: `sentimento_label`
3. Preview: `sentimento_label` deve ser `positivo`, `misto` ou `negativo`. Compare com `sentimento_esperado`.

### 5.4 Join tickets × pedidos da cidade

Inner join em `pedido_id` com `filtra_cidade` (não há Filter de `confirmado`). Cada linha é um ticket com o pedido ao lado (`produto`, `cidade`, `receita`, `status`, `canal_venda`).

1. **Join** `experiencia_pedido`.
2. Entrada 1 (left): `extrai_sentimento`. Entrada 2 (right): `filtra_cidade`.
3. Condição: `pedido_id` = `pedido_id`.
4. Join type: **Inner join**.
5. Match case: off.

Tickets cujo pedido falhou no Guardrails (ex.: Latte `1014` com valor negativo) ou está fora da cidade do parâmetro não entram.

### 5.5 Select — colunas finais

1. Adicione **Select**. Nome: `colunas_finais`.
2. Modo passthrough (mantém o restante).
3. Desmarque: `sentimento_esperado`, `right_pedido_id`, `right_cliente_id`, `right_status`.


---
## Módulo 6 — Output, Group e mapa do canvas

### 6.1 Output — gravar no Unity Catalog

**Output** escreve tabela, materialized view ou arquivo em Volume. Write mode: Overwrite / Append / Merge.

1. A partir de `colunas_finais`, adicione **Output**.
2. Output type: **Table**.
3. Table name: `{seu_prefixo}aurora_experiencia_pedido`
4. Location: seu catálogo.schema
5. Write mode: **Overwrite**
6. Clique em **Run** (não só o preview). Se a tabela não existir, o operador cria.

Opcional: um segundo Output tipo **File** (CSV no Volume) para mostrar split / header.

### 6.2 Groups

| Group | Operadores |
|---|---|
| `Fontes` | 3× Source |
| `Qualidade e canais` | Unique, Prepare, Guardrails, Filter, Combine |
| `Pipeline de sentimento` | Filter texto, AI Function, Prepare (rótulo), Join `pedido_id`, Select finais, Output |

### Mapa final

```
clientes ─ Unique ─ Prepare ───────────────────────┐
                                         ├─ Join cliente_id ─ Select ─ SQL(:cidade) ─┐
pedidos ─ Unique ─ Prepare ─ Guardrails ─ Filter                                     │
                         │ Included ─ Prepare ─┐                                     ├─ Join pedido_id ─ Select ─ Output
                         │ Excluded ─ Prepare ─┴ Combine                             │
                                                                                     │
tickets ─ Filter ─ AI classify ─ Prepare ────────────────────────────────────────────┘
```


---
## Módulo 7 — Genie Code *(opcional)*

Em um **novo** visual data prep, descreva:

> Usando as tabelas {prefixo}clientes, {prefixo}pedidos e {prefixo}tickets_suporte: deduplicar, padronizar canal, validar status e valor, unir ONLINE e LOJA, juntar pedidos com clientes em cliente_id, filtrar cidade. Nos tickets, classificar sentimento em positivo/misto/negativo, juntar com o pedido em pedido_id, selecionar colunas e escrever em {prefixo}aurora_experiencia_pedido.

Confira cada operador gerado. Ver [Genie Code](https://docs.databricks.com/aws/en/designer/genie-code).


---
## Validação final (SQL)

Depois do **Run** do Output, confira a tabela ouro.


In [0]:
%sql
SELECT *
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}aurora_experiencia_pedido`
ORDER BY produto, sentimento;


In [0]:
%sql
SELECT produto, sentimento, COUNT(*) AS qtd, ROUND(SUM(receita), 2) AS receita
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}aurora_experiencia_pedido`
GROUP BY produto, sentimento
ORDER BY produto, sentimento;


---
## Módulo 8 — Dashboard AI/BI *(opcional)*

Depois do **Run** do Output, crie um [AI/BI Dashboard](https://docs.databricks.com/aws/en/dashboards/) em cima da tabela `{seu_prefixo}aurora_experiencia_pedido`.

1. **New** → **Dashboard**.
2. Adicione a tabela como dataset.
3. Abra o **Genie Code** e cole:

> Crie um dashboard de experiência do pedido para a Aurora Café usando a tabela `{seu_prefixo}aurora_experiencia_pedido`.
> Cada linha é um ticket de suporte junto do pedido (produto, receita, status, canal_venda, cidade, segmento).
> O sentimento do comentário está em `sentimento_label` (`positivo`, `misto`, `negativo`).
>
> Inclua:
> - KPIs: quantidade de tickets, receita total, % de tickets negativos
> - Barras de quantidade e receita por `produto` e `sentimento_label`
> - Comparação de sentimento por `canal_venda` e por `cidade`
> - Tabela com `pedido_id`, `produto`, `nome`, `comentario`, `sentimento_label`, `receita`, `status`
>
> Título: Aurora Café — experiência do pedido. Filtros em cidade, canal_venda, produto e sentimento_label.
